In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 7. Local Research Paper Explainer

## What We're Building
A tool that takes an academic paper and produces:
- Plain-English summary of the key contribution
- Simplified explanation of the methodology
- Key findings in bullet points
- Why this paper matters (practical implications)

**You will learn:**
- LlamaIndex for PDF processing
- Section-aware document handling
- Simplification prompting strategies
- Progressive explanation (ELI5 → expert)

**Stack:** Ollama, LlamaIndex, PDF parsing

In [ ]:
# !pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama pypdf

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings, Document
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from pathlib import Path

Settings.llm = Ollama(model="qwen3:8b", base_url="http://localhost:11434", request_timeout=120)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text", base_url="http://localhost:11434")
Settings.chunk_size = 1024
print("LlamaIndex configured!")

## Step 1 — Create a Sample Research Paper (Simulated)

In [ ]:
# Simulate a research paper abstract and sections
paper_text = """
Title: Attention Is All You Need

Abstract:
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer, based
solely on attention mechanisms, dispensing with recurrence and convolutions entirely.
Experiments on two machine translation tasks show these models to be superior in
quality while being more parallelizable and requiring significantly less time to train.
Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task,
improving over the existing best results by over 2 BLEU. On the WMT 2014
English-to-French translation task, our model establishes a new single-model
state-of-the-art BLEU score of 41.8.

1. Introduction:
Recurrent neural networks, long short-term memory and gated recurrent neural networks
have been firmly established as state of the art approaches in sequence modeling and
transduction problems. Recurrent models typically factor computation along the symbol
positions of the input and output sequences. This inherently sequential nature precludes
parallelization within training examples, which becomes critical at longer sequence
lengths.

The Transformer uses multi-head self-attention to compute representations of its input
and output without using sequence-aligned RNNs or convolution. This allows significantly
more parallelization and can reach a new state of the art in translation quality.

2. Model Architecture:
The Transformer follows an encoder-decoder structure using stacked self-attention and
point-wise fully connected layers. The encoder maps an input sequence of symbol
representations to a sequence of continuous representations. The decoder then generates
an output sequence of symbols one element at a time.

Key components:
- Multi-Head Attention: Instead of performing a single attention function, we found it
  beneficial to linearly project the queries, keys and values multiple times. This allows
  the model to jointly attend to information from different representation subspaces.
- Positional Encoding: Since the model contains no recurrence, we inject information about
  the relative or absolute position using sinusoidal positional encodings.
- Feed-Forward Networks: Each layer contains a fully connected feed-forward network applied
  to each position separately and identically.

3. Results:
On the WMT 2014 English-to-German translation task, the big transformer model outperforms
the best previously reported models including ensembles by more than 2.0 BLEU, establishing
a new state-of-the-art BLEU score of 28.4. Training took 3.5 days on 8 P100 GPUs.
On the WMT 2014 English-to-French task, the big model achieves a BLEU score of 41.0,
outperforming all the previously published single models, at less than 1/4 the training
cost of the previous state-of-the-art model.
"""

# Create document
doc = Document(text=paper_text, metadata={"title": "Attention Is All You Need", "type": "research_paper"})
index = VectorStoreIndex.from_documents([doc])
query_engine = index.as_query_engine(similarity_top_k=5)
print(f"Paper indexed! ({len(paper_text)} characters)")

## Step 2 — Generate Explanations at Different Levels

In [ ]:
from llama_index.core import PromptTemplate

# ELI5 explanation
eli5_prompt = """Based on the research paper content below, explain the main idea
as if you're talking to a smart 12-year-old. No jargon. Use analogies.

Context: {context_str}

Question: {query_str}

Simple explanation:"""

query_engine.update_prompts({"response_synthesizer:text_qa_template": PromptTemplate(eli5_prompt)})
eli5 = query_engine.query("What is this paper about and why does it matter?")
print("=== ELI5 Explanation ===")
print(eli5)

In [ ]:
# Practitioner explanation
practitioner_prompt = """Based on the research paper content below, explain the key
contribution for an ML practitioner. Include:
1. What problem does it solve?
2. What's the core technical insight?
3. Key results and improvements
4. Practical implications

Context: {context_str}

Question: {query_str}

Practitioner explanation:"""

query_engine.update_prompts({"response_synthesizer:text_qa_template": PromptTemplate(practitioner_prompt)})
practitioner = query_engine.query("Explain the key contribution and results")
print("=== Practitioner Explanation ===")
print(practitioner)

In [ ]:
# Key findings extraction
findings_prompt = """Extract the key findings from this research paper.
Format as structured bullet points.

Context: {context_str}

Question: {query_str}

Key findings:"""

query_engine.update_prompts({"response_synthesizer:text_qa_template": PromptTemplate(findings_prompt)})
findings = query_engine.query("What are the key findings and results?")
print("=== Key Findings ===")
print(findings)

In [ ]:
# Full paper explainer function
def explain_paper(paper_text: str, title: str = "Research Paper") -> dict:
    """Generate multi-level explanations of a research paper."""
    doc = Document(text=paper_text, metadata={"title": title})
    idx = VectorStoreIndex.from_documents([doc])
    qe = idx.as_query_engine(similarity_top_k=5)

    prompts = {
        "eli5": "Explain this paper for a non-expert. Use simple language and analogies.",
        "summary": "Provide a 3-sentence summary of the key contribution.",
        "findings": "List the key findings as bullet points.",
        "implications": "What are the practical implications of this work?",
    }

    results = {}
    for key, question in prompts.items():
        results[key] = str(qe.query(question))
        print(f"\n{'='*40}")
        print(f"{key.upper()}")
        print(f"{'='*40}")
        print(results[key])

    return results

results = explain_paper(paper_text, "Attention Is All You Need")

## Summary & Next Steps

**What you learned:**
- ✅ LlamaIndex document indexing and query engines
- ✅ Custom prompt templates for different explanation levels
- ✅ Section-aware analysis of structured documents
- ✅ Progressive explanation strategies (ELI5 → expert)

**Next steps:**
- Load real PDFs with `PyPDFLoader` or LlamaIndex's `PDFReader`
- Compare explanations across different models
- Add figure/table description capabilities

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings

# Set LlamaIndex to use local Ollama
Settings.llm = Ollama(model="qwen3.5:9b", request_timeout=120.0)
Settings.embed_model = "local:BAAI/bge-small-en-v1.5"

# documents = SimpleDirectoryReader('data').load_data()
# index = VectorStoreIndex.from_documents(documents)
# query_engine = index.as_query_engine()
# response = query_engine.query("What are the main insights?")
print("LlamaIndex configured for local RAG execution!")
